# Federal Dollars: Out and Back In
How much federal tax is collected in California, and how much federal money comes back as benefits, pay and funds to California governments? This page shows several pieces of that picture. **It is not a complete balance of payments** and should not be read as one.

**What is here**
- **Taxes out:** federal tax collections attributed to California by the IRS (Data Book Table 5, federal fiscal years ending September 30). The IRS assigns each payment to the state where it was filed or deposited, so a multi-state employer's withholding or corporate tax can land in its home state. Includes individual income and payroll taxes, corporate income tax, excise, estate and gift taxes.
- **Money back in, to people:** government benefits paid to Californians and federal civilian and military pay (Bureau of Economic Analysis, calendar years). Programs that are federal by design are marked federal; Medicaid and unemployment insurance are shared with the state, and BEA's "other" benefits mix levels of government.
- **Money back in, to governments:** federal funds spent by the state, and federal aid received by counties, cities and school districts (from the state, SCO and Census data on the earlier pages).

**What is not here:** federal contracts and procurement (defense, NASA, research), most grants to universities and businesses, interest on federal debt, tax breaks and deductions, and anything paid to Californians that BEA does not classify as a government transfer. Those omissions mean the "back in" side is understated. Social Security and Medicare are funded by dedicated taxes and are not designed to balance state by state; California's age mix, income and industries all affect where taxes and benefits land.

**Timing:** IRS years are federal fiscal years (October to September); BEA years are calendar years. They are lined up by the year number, which overlaps about nine months. All dollar values below use the same CPI and population series as the rest of the site.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, line_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
import numpy as np
fc = pd.read_csv(PROCESSED / "federal_collections.csv")
fb = pd.read_csv(PROCESSED / "federal_benefits.csv")
gov = pd.read_csv(PROCESSED / "federal_govt_receipts.csv")
IRS = "Source: IRS Data Book Table 5 (gross collections by state, federal fiscal years)"
d = fc.melt(id_vars="fy", value_vars=["individual_employment_k", "business_income_k", "other_k"], var_name="type", value_name="amount_k")
d["type"] = d.type.map({"individual_employment_k": "Individual income and payroll taxes", "business_income_k": "Corporate (business) income tax",
                        "other_k": "Excise, estate, gift and other"})
stacked_area_toggle(add_views(d), "type", "Federal taxes collected in California",
                    IRS + ". FY2023 and FY2024 are distorted by postponed filing deadlines (see below).",
                    order=["Individual income and payroll taxes", "Corporate (business) income tax", "Excise, estate, gift and other"],
                    breaks={2023.5: "Filing deadlines postponed"}).show()
sh = (fc.set_index("fy").total_k / fc.set_index("fy").us_total_k * 100)
print("California share of US federal collections (%), selected years:", {int(k): round(v, 1) for k, v in sh.loc[[1998, 2008, 2018, 2022, 2023, 2024, 2025]].items()})

California share of US federal collections (%), selected years: {1998: 11.3, 2008: 11.6, 2018: 13.2, 2022: 14.2, 2023: 12.1, 2024: 15.8, 2025: 14.4}


California's share of national collections dips in FY2023 and jumps in FY2024. This is consistent with the IRS postponing 2022 filing and payment deadlines for California (winter storms) into the fall of 2023, which moved payments from FY2023 into FY2024; the series is shown as published, so read those two years together.

## Federal benefits and pay received by Californians
BEA personal current transfer receipts from governments, plus pay for federal civilian employees and the military, calendar years. Medicaid, unemployment insurance and the 'other' group include state and local money.

In [2]:
d = fb.copy()
d["fy"] = d.year
order = ["Social Security", "Medicare", "Veterans' benefits", "Supplemental Security Income", "SNAP (food stamps)", "Earned Income Tax Credit", "Military medical insurance",
         "Federal civilian pay", "Military pay", "Medicaid (federal + state share)", "Unemployment insurance (state + federal)", "Other government benefits (mixed levels)"]
fig = stacked_area_toggle(add_views(d[["fy", "channel", "amount_k"]]), "channel", "Federal benefits and pay received in California",
                          "BEA, calendar years. The last three layers include state and local money. Source: Bureau of Economic Analysis (SAINC35, SAINC6)", order=order)
fig.update_layout(xaxis_title="Calendar year")
fig.show()

## Taxes out versus benefits and pay back in, per person
Three measures in constant dollars per Californian. 'Federal programs and pay' counts only programs that are federal by design. The dashed line adds Medicaid and unemployment insurance, which are shared with the state, so it overstates the federal share of those programs. None of the lines includes procurement or grants to organizations.

In [3]:
fed = fb[fb.federal_only].groupby("year").amount_k.sum()
shared = fb[fb.channel.isin(["Medicaid (federal + state share)", "Unemployment insurance (state + federal)"])].groupby("year").amount_k.sum()
tax = fc.set_index("fy").total_k
yrs = sorted(set(tax.index) & set(fed.index))
rows = []
for name, s in [("Federal taxes collected", tax), ("Federal programs and pay", fed), ("Plus Medicaid and unemployment (shared with state)", fed + shared)]:
    rows.append(pd.DataFrame({"fy": yrs, "group": name, "amount_k": s.reindex(yrs).values}))
d = add_views(pd.concat(rows))
fig = line_toggle(d, "group", "Federal taxes out and benefits back in, per person",
                  "Taxes: IRS federal fiscal year. Benefits and pay: BEA calendar year, same year number. Not a complete balance of payments.",
                  order=["Federal taxes collected", "Federal programs and pay", "Plus Medicaid and unemployment (shared with state)"],
                  breaks={2023.5: "Filing deadlines postponed"}, default=2,
                  dashes={"Plus Medicaid and unemployment (shared with state)": "dash"})
fig.update_layout(xaxis_title="Year")
fig.show()
r = pd.DataFrame({"Federal programs and pay": fed.reindex(yrs) / tax.reindex(yrs) * 100,
                  "... plus Medicaid and unemployment insurance": (fed + shared).reindex(yrs) / tax.reindex(yrs) * 100})
fig = go.Figure()
for i, c in enumerate(r.columns):
    fig.add_trace(go.Scatter(x=r.index, y=r[c], name=c, mode="lines+markers", line=dict(color=PALETTE[[1, 2][i]], dash="solid" if i == 0 else "dash"), marker=dict(size=4), hovertemplate="%{y:.0f}%"))
fig.update_layout(yaxis_title="% of federal taxes collected in California", yaxis_ticksuffix="%", xaxis_title="Year")
base_layout(fig, "Benefits and pay as a share of federal taxes collected", "Selected federal money to people and pay, as a percent of IRS collections. Distorted in 2023-24 by postponed deadlines. Excludes contracts and grants.")
fig.add_vline(x=2023.5, line_dash="dot", line_color="grey", annotation_text="Filing deadlines postponed", annotation_position="bottom left")
fig.show()

## Federal funds received by California governments
Federal money received by the state, counties, cities and school districts. **These lines overlap and must not be added**: state spending of federal funds includes money passed to counties and schools, and much county federal aid is public assistance and health programs that also run through the state budget. Counties and cities start in 2003 (SCO).

In [4]:
d = add_views(gov.dropna())
fig = line_toggle(d, "recipient", "Federal funds received by California governments",
                  "Overlapping series, not additive. State = federal funds spent (LAO / DOF); counties and cities = SCO federal aid; schools = Census federal revenue",
                  order=["State government (federal funds spent)", "Counties", "School districts", "Cities"], breaks={2016.5: "SCO reporting format changes"})
fig.show()

The state's federal funds jump in FY2020-21 almost entirely because of two areas: the Employment Development Department, which pays unemployment benefits ($141B of federal funds that year versus $7B in 2019), and K-12 education ($27B versus $8B). Neither is a lasting change in the level of federal support; see the by-area chart below.

## Federal funds in the state budget
Federal money the state spends, by area (all years from the state's historical expenditure data). The estimate shading marks budget proposals.

In [5]:
e = pd.read_csv(PROCESSED / "state_expenditures.csv")
f = e[e.fund == "Federal Funds"].groupby(["fy", "area"], as_index=False).amount_k.sum()
top = f.groupby("area").amount_k.sum().sort_values(ascending=False)
keep = list(top.head(7).index)
f["area"] = f.area.where(f.area.isin(keep), "All other areas")
stacked_area_toggle(add_views(f.groupby(["fy", "area"], as_index=False).amount_k.sum()), "area", "Federal funds spent by the state, by area",
                    "State historical expenditures, federal fund type. Source: Legislative Analyst's Office / Department of Finance").show()
tot = e.groupby(["fy", "fund"]).amount_k.sum().unstack()
share = (tot["Federal Funds"] / tot.sum(axis=1) * 100)
fig = go.Figure(go.Scatter(x=share.index, y=share, mode="lines+markers", line=dict(color=PALETTE[0]), marker=dict(size=4), hovertemplate="%{y:.1f}%"))
fig.update_layout(yaxis_title="% of total state spending", yaxis_ticksuffix="%", xaxis_title="Fiscal year (ending June 30)", showlegend=False)
base_layout(fig, "Federal funds as a share of state spending", "Federal funds / (general, special, bond and federal funds). Source: LAO / DOF", height=460)
fig.add_vrect(x0=LAST_ACTUAL_FY + 0.5, x1=share.index.max(), fillcolor="grey", opacity=0.12, line_width=0, annotation_text="estimate", annotation_position="top left")
fig.show()
y = e[e.fy == LAST_ACTUAL_FY].pivot_table(index="area", columns="fund", values="amount_k", aggfunc="sum").fillna(0)
sh = (y["Federal Funds"] / y.sum(axis=1) * 100).sort_values()
fig = go.Figure(go.Bar(y=sh.index, x=sh, orientation="h", marker_color=PALETTE[0], hovertemplate="%{y}: %{x:.0f}%<extra></extra>"))
fig.update_layout(xaxis_title="% of the area's spending paid from federal funds", xaxis_ticksuffix="%", showlegend=False, hovermode="closest")
base_layout(fig, f"Share of each state spending area funded by federal funds, FY{LAST_ACTUAL_FY}", "Federal funds / all funds in the area. Source: LAO / DOF", height=520)
fig.update_layout(margin=dict(l=260, r=20, t=90, b=50))
fig.show()